# Prior sensitivity and model extensions: data and corner plots

Loads the October 2026 runs (`best_prior/` on the cluster, or a local copy) and their priors, and draws plain corner plots for you to style.

* **Prior sensitivity (appendix):** `main_joint` (fiducial prior), `restrict_joint` (no widening), `flat_joint` (no simulation information), `tpool_joint` (fiducial with the pooled $t_\ast$ width).
* **Extensions (Sec. 5.1):** `ext_fstar_z`, `ext_alpha_star_z`, `ext_slope*`, `ext_sigma_z`, `ext_sigma_mh`.

Each run folder needs `post_equal_weights.dat`, `run_config.json` (parameter order) and `stats.dat` (evidence). Priors are the `prior_<tag>.json` files from `build_priors.py`.

In [ ]:
import json, os, re
import numpy as np
import matplotlib.pyplot as plt
import corner
from scipy.stats import truncnorm

RUNS_DIR = "/lustre/astro/ivannik/UVLF_clust_runs/best_prior"            # or a local copy
PRIORS_DIR = "/groups/astro/ivannik/programs/JWST-Inference/priors_generated"

LABELS = {
    "fstar_norm": r"$\log_{10} f_{\ast,0}$", "sigma_SHMR": r"$\sigma_{\rm SHMR}$",
    "t_star": r"$t_\ast$", "alpha_star_low": r"$\alpha_\ast$",
    "sigma_SFMS_norm": r"$\sigma_{\rm SFMS,0}$", "a_sig_SFR": r"$a_{\sigma,\rm SFMS}$",
    "M_knee": r"$\log_{10} M_{\rm knee}$", "sigma_sfr_10": r"$\sigma_{\rm SFR,10}$",
    "alpha_fstar_z": r"$\alpha_{f_\ast,z}$", "alpha_star_z": r"$\alpha_{\ast,z}$",
    "slope_SFR": r"slope$_{\rm SFMS}$", "alpha_sigma_shmr_z": r"$\alpha_{\sigma,{\rm SHMR},z}$",
    "a_sig_SHMR": r"$a_{\sigma,\rm SHMR}$",
}

In [ ]:
def load_run(name):
    d = os.path.join(RUNS_DIR, name)
    with open(os.path.join(d, "run_config.json")) as f:
        params = json.load(f)["params"]
    post = np.loadtxt(os.path.join(d, "post_equal_weights.dat"))
    with open(os.path.join(d, "stats.dat")) as f:
        lnz, lnz_err = map(float, re.search(
            r"Global Log-Evidence\s*:\s*([-\d.E+]+)\s*\+/-\s*([\d.E+-]+)", f.read()).groups())
    return dict(params=params, samples=post[:, :len(params)], lnL=post[:, len(params)],
                lnZ=lnz, lnZ_err=lnz_err)


def load_prior(tag):
    with open(os.path.join(PRIORS_DIR, f"prior_{tag}.json")) as f:
        return json.load(f)


def prior_samples(spec, params, n=200_000, seed=0):
    """Draws from a prior as mcmc.py samples it: truncated, diagonal Gaussian."""
    rng = np.random.default_rng(seed)
    out = []
    for p in params:
        j = spec["params"].index(p)
        m, sd = spec["mean"][j], spec["effective_sd"][j]
        lo, hi = spec["sampling_limits"][j]
        out.append(truncnorm.rvs((lo - m) / sd, (hi - m) / sd, loc=m, scale=sd,
                                 size=n, random_state=rng))
    return np.array(out).T


def columns(run, params):
    return run["samples"][:, [run["params"].index(p) for p in params]]

## Prior sensitivity (appendix)

In [ ]:
PRIOR_RUNS = {           # run folder: prior tag
    "main_joint": "best",
    "restrict_joint": "restrictive",
    "flat_joint": "flat",
    "tpool_joint": "best_tstar_pooled",
}
runs = {r: load_run(r) for r in PRIOR_RUNS}
priors = {r: load_prior(t) for r, t in PRIOR_RUNS.items()}

for r, d in runs.items():
    print(f"{r:16s} lnZ = {d['lnZ']:.3f} +- {d['lnZ_err']:.3f}   "
          f"dlnZ vs main_joint = {d['lnZ'] - runs['main_joint']['lnZ']:+.2f}")

In [ ]:
# Parameters shown (M_knee is fixed; sigma_sfr_10 has the same prior in all runs).
SHOW = ["fstar_norm", "sigma_SHMR", "t_star", "alpha_star_low", "sigma_SFMS_norm", "a_sig_SFR"]
colors = {"main_joint": "C3", "restrict_joint": "C7", "flat_joint": "C0", "tpool_joint": "C2"}

fig = None
for r in ["flat_joint", "restrict_joint", "tpool_joint", "main_joint"]:   # drawn back to front
    fig = corner.corner(columns(runs[r], SHOW), fig=fig, color=colors[r],
                        labels=[LABELS[p] for p in SHOW], levels=(0.68, 0.95),
                        plot_datapoints=False, plot_density=False, smooth=1.0,
                        hist_kwargs=dict(density=True))
plt.show()

In [ ]:
# Same, with the fiducial prior overplotted (dashed) to show which parameters are data- vs prior-driven.
fig = corner.corner(prior_samples(priors["main_joint"], SHOW), color="0.5",
                    labels=[LABELS[p] for p in SHOW], levels=(0.68, 0.95),
                    plot_datapoints=False, plot_density=False, smooth=1.0,
                    hist_kwargs=dict(density=True, ls="--"), contour_kwargs=dict(linestyles="--"))
corner.corner(columns(runs["main_joint"], SHOW), fig=fig, color="C3", levels=(0.68, 0.95),
              plot_datapoints=False, plot_density=False, smooth=1.0, hist_kwargs=dict(density=True))
plt.show()

In [ ]:
# Numbers for the text: median and 68% interval, and posterior/prior width ratio.
for p in SHOW:
    row = f"{p:16s}"
    for r in PRIOR_RUNS:
        x = columns(runs[r], [p])[:, 0]
        lo, m, hi = np.percentile(x, [16, 50, 84])
        row += f" | {r[:8]} {m:7.3f} [{lo:6.3f},{hi:6.3f}]"
    pr = prior_samples(priors["main_joint"], [p])[:, 0]
    x = columns(runs["main_joint"], [p])[:, 0]
    row += f" | width ratio (fiducial) {np.diff(np.percentile(x, [16, 84]))[0] / np.diff(np.percentile(pr, [16, 84]))[0]:.2f}"
    print(row)

## sigma_UV decomposition (`scatter_contribution_sfr10*.npz`)

In [ ]:
for r in ["main_joint", "restrict_joint"]:
    for f in ["scatter_contribution_sfr10.npz", "scatter_contribution_sfr10_muv-18.npz"]:
        path = os.path.join(RUNS_DIR, r, f)
        if not os.path.exists(path):
            continue
        d = np.load(path)
        out = f"{r:15s} Muv={float(d['target_muv']):.0f}  sigma_UV {np.median(d['sigma_full']):.2f}"
        for k in ["frac_no_SFMS", "frac_no_SHMR", "frac_no_sfr10"]:
            v = d[k][np.isfinite(d[k])]
            lo, m, hi = np.percentile(100 * v, [16, 50, 84])
            out += f" | {k[8:]} {m:.0f}% [{lo:.0f},{hi:.0f}]"
        print(out)

## Model extensions (Sec. 5.1)

In [ ]:
EXT_RUNS = {   # run folder: (prior tag, extension parameter, null value)
    "ext_fstar_z": ("best_fstar_z", "alpha_fstar_z", 0.0),
    "ext_alpha_star_z": ("best_alpha_star_z", "alpha_star_z", 0.0),
    "ext_slope": ("best_slope_SFR", "slope_SFR", 1.0),          # adjust folder name for the pivot version
    "ext_sigma_z": ("best_sigma_star_z", "alpha_sigma_shmr_z", 0.0),
    "ext_sigma_mh": ("best_sigma_star_Mh", "a_sig_SHMR", 0.0),
}
ext = {r: load_run(r) for r in EXT_RUNS if os.path.isdir(os.path.join(RUNS_DIR, r))}
ext_priors = {r: load_prior(EXT_RUNS[r][0]) for r in ext}

lnz0 = runs["main_joint"]["lnZ"]
for r, d in ext.items():
    tag, p, null = EXT_RUNS[r]
    x = columns(d, [p])[:, 0]
    lo, m, hi = np.percentile(x, [16, 50, 84])
    pr = prior_samples(ext_priors[r], [p])[:, 0]
    print(f"{r:18s} {p:19s} post {m:+.3f} [{lo:+.3f},{hi:+.3f}]  prior {np.median(pr):+.3f} "
          f"[{np.percentile(pr, 16):+.3f},{np.percentile(pr, 84):+.3f}]  "
          f"P(<null) post/prior {np.mean(x < null):.2f}/{np.mean(pr < null):.2f}  dlnZ {d['lnZ'] - lnz0:+.2f}")

In [ ]:
# Corner plot for one extension: base parameters + the extension parameter, prior dashed.
r = "ext_sigma_z"
p_ext = EXT_RUNS[r][1]
show = SHOW[:3] + [p_ext]
fig = corner.corner(prior_samples(ext_priors[r], show), color="0.5", labels=[LABELS[p] for p in show],
                    levels=(0.68, 0.95), plot_datapoints=False, plot_density=False, smooth=1.0,
                    hist_kwargs=dict(density=True, ls="--"), contour_kwargs=dict(linestyles="--"))
corner.corner(columns(ext[r], show), fig=fig, color="C3", levels=(0.68, 0.95),
              plot_datapoints=False, plot_density=False, smooth=1.0, hist_kwargs=dict(density=True))
plt.show()